# planner 메모리 수정 검증 (실제 RunPod)

최근 작업한 두 가지를 라이브로 확인한다.

- **#4 롤링 요약 배선** — follow_up 으로 history 가 길어지면(>3턴) 오래된 턴이
  LLM 요약 1줄로 접히고 `memory_summary` 가 채워진다. history 무한증가도 같이 막힌다.
  (`agents/todo_creation/planner/memory.py::fold_history` + `QwenLLM.summarize_history`)
- **#1 MemorySaver 누수 차단** — in-process LRU 로 thread 수 상한을 두고 초과 시
  가장 오래된 thread 를 evict 한다. (`pipeline.py::_touch_thread`) — 외부 호출 없음.

> ⚠️ **#4 셀은 유료 RunPod 호출** — `.env` 의 `RUNPOD_API_KEY` /
> `RUNPOD_PLANNER_ENDPOINT_URL` 필요. 콜드스타트(~100s)+다중 호출이라 셀당 수십초~분.
> #1 셀은 인메모리라 공짜·즉시.

**fold 발동 조건**: history 가 trigger(3)를 넘어야 한다. 그래프는 follow_up 을 최대
2라운드(=4턴)로 제한하므로 **되묻기가 2번 일어나야** memory_summary 가 채워진다.
1번만 되묻고 바로 계획을 생성하면 fold 가 안 일어나는 게 정상(§1 가이드 참고).

## 0. 셋업 — .env 로드 + RunPod planner/base 어댑터 구성

In [ ]:
import sys, os
from datetime import date, datetime
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# .env 직접 로드(이미 설정된 값은 보존). 시크릿은 하드코딩하지 않는다.
env_path = ROOT / '.env'
if env_path.exists():
    for line in env_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

url = os.environ.get('RUNPOD_PLANNER_ENDPOINT_URL')
key = os.environ.get('RUNPOD_API_KEY')
assert url and key, 'RUNPOD_PLANNER_ENDPOINT_URL / RUNPOD_API_KEY 가 .env 에 필요'
print('endpoint set:', bool(url), '| key set:', bool(key))

In [ ]:
from adapters.todo_creation.runpod_llm import RunPodQwenLLM
from agents.todo_creation.planner.pipeline import PlannerPorts, run, get_debug_state
from agents.todo_creation.schemas import PlannerInput

# llm=planner LoRA, classifier/validator=base. follow_up 요약은 question_llm(=classifier)
# 이 담당하므로 base 어댑터가 summarize_history 를 친다(deps.build_todo_planner_ports 와 동일 배선).
planner_llm = RunPodQwenLLM(endpoint_url=url, api_key=key, adapter='planner')
base_llm = RunPodQwenLLM(endpoint_url=url, api_key=key, adapter='base')
ports = PlannerPorts(llm=planner_llm, classifier=base_llm, validator=base_llm)
TODAY = date.today(); NOW = datetime.now()
print('ready. today =', TODAY)

In [ ]:
def show(tag, thread_id):
    """턴마다 메모리 상태를 찍는다 — history 가 bounded 한지, memory_summary 가 채워졌는지."""
    st = get_debug_state(thread_id=thread_id, ports=ports)
    print(f'[{tag}] history_turns={st["history_turns"]} '
          f'recent={len(st["recent_turns"])} follow_up_count={st["follow_up_count"]}')
    print('   memory_summary:', st.get('memory_summary'))
    return st

## 1. #4 롤링 요약 — 다중 턴으로 follow_up 2회 유도

일부러 모호하게 시작해 2번 되묻게 만든다. 각 턴 후 `show()` 로 메모리 상태를 본다.
메시지를 바꿔가며 재현 가능. (각 셀 ⚠️ 유료 호출)

In [ ]:
# turn1 — 매우 모호 → 되물어야 정상(FollowUpResult)
TID = 'mem-verify-1'
r1 = await run(PlannerInput(user_id='memv', message='뭔가 준비 좀 해야 하는데', today=TODAY,
                            thread_id=None), ports=ports, now=NOW)  # ⚠️ 유료
TID = r1.thread_id
print('turn1:', type(r1).__name__, '|', getattr(r1, 'question', None))
show('after turn1', TID)  # history 2턴, memory_summary None 예상

In [ ]:
# turn2 — 여전히 불완전한 답 → 2번째 되물음을 유도(FollowUpResult)
r2 = await run(PlannerInput(user_id='memv', message='시험 준비요', today=TODAY,
                            thread_id=TID), ports=ports, now=NOW)  # ⚠️ 유료
print('turn2:', type(r2).__name__, '|', getattr(r2, 'question', None))
show('after turn2', TID)  # 되물었다면 history 4턴>trigger → ✅ memory_summary 채워짐

In [ ]:
# turn3 — 답을 채워 계획 생성으로 보냄
r3 = await run(PlannerInput(user_id='memv', message='토익이고 다음달 시험이에요. 평일 저녁 1시간.',
                            today=TODAY, thread_id=TID), ports=ports, now=NOW)  # ⚠️ 유료
print('turn3:', type(r3).__name__)
st = show('after turn3', TID)

ms = st.get('memory_summary')
if ms and ms.get('text'):
    print('\n✅ #4 검증 성공 — memory_summary 가 채워짐:')
    print('   ', ms['text'])
    assert st['history_turns'] <= 3, 'history 가 fold 로 bounded 여야 함'
    print('✅ history bounded:', st['history_turns'], '턴 (요약 1 + 최근 2 이하)')
else:
    print('\n⚠️ memory_summary 비어있음 — 되묻기가 2회 미만이라 fold 가 안 일어남.')
    print('   turn1/turn2 가 둘 다 FollowUpResult 였는지 확인하고, 아니면 더 모호한')
    print('   메시지로 되묻기를 2번 유도해 재실행하라.')

### 판정 가이드 (#4)
- `memory_summary.text` 가 채워지고 한국어 요약이면 → **요약 LLM 호출이 살아있음**(배선 OK).
- `history_turns` 가 fold 후 3 이하로 유지 → **무한증가 차단 OK**.
- 요약 텍스트가 앞 대화의 목표/조건을 담고 있는지 눈으로 확인(환각 아님).
- memory_summary 가 계속 비면: 모델이 1회만 되묻고 생성함 → 더 모호한 입력으로 2회 유도.

## 2. #1 MemorySaver 누수 차단 (인메모리 · 공짜)

`_touch_thread` 가 한도를 넘으면 가장 오래된 thread 를 evict 하고 `delete_thread` 를
호출하는지 본다. RunPod 호출 없음.

In [ ]:
from unittest.mock import Mock
from collections import OrderedDict
from agents.todo_creation.planner import pipeline as P

# 한도 3 으로 줄이고 evict 를 관찰(모듈 전역만 임시 변경 — 노트북 한정).
P._MAX_LIVE_THREADS = 3
P._live_threads = OrderedDict()
spy = Mock()
P._GRAPH.checkpointer.delete_thread = spy

for tid in ['a', 'b', 'c', 'd', 'e']:
    P._touch_thread(tid)
    print('touch', tid, '→ live:', list(P._live_threads))

print('\nevict 호출:', [c.args[0] for c in spy.call_args_list])
assert list(P._live_threads) == ['c', 'd', 'e'], '최신 3개만 남아야 함'
assert [c.args[0] for c in spy.call_args_list] == ['a', 'b'], 'a,b 가 evict 돼야 함'
print('✅ #1 검증 성공 — 상한 초과분(a,b) evict, delete_thread 호출됨')

### 판정 가이드 (#1)
- 상한(3)을 넘기는 순간 가장 오래된 thread 가 `_live_threads` 에서 빠지고
  `delete_thread` 가 그 id 로 호출되면 → **누수 차단 OK**.
- 실제 운영 기본값은 `_MAX_LIVE_THREADS=500`. evict 된 thread 는 재시작처럼 새 상태로
  degrade(기존 단일워커 동작과 동일) — 영속화는 안 한 결정대로다.